In [56]:
import os, sys
import hashlib
import ctypes
from tqdm import tqdm
from utils import *
import tree_sitter
from tree_sitter import Language, Parser, QueryCursor, Query

In [3]:
import ctypes

lib = ctypes.cdll.LoadLibrary('/home/chentianyu/verus.so')
lang_func = lib.tree_sitter_verus
lang_func.restype = ctypes.c_void_p
ptr = lang_func()

lang = Language(ptr)
parser = Parser(lang)
tree = parser.parse(b'fn main() {}')

/tmp/ipykernel_3677354/2203168636.py:8: DeprecationWarning: int argument support is deprecated
  lang = Language(ptr)


In [157]:
class verus_parser:
    def __init__(self, language_path:str) -> None:
        lib = ctypes.cdll.LoadLibrary(language_path)
        lang_func = lib.tree_sitter_verus
        lang_func.restype = ctypes.c_void_p
        ptr = lang_func()

        self.language = Language(ptr)
        self.parser = Parser(self.language)

    def capture_query(self, program:tree_sitter.Node, query_str:str):
        query = Query(self.language, query_str)
        query_curser = QueryCursor(query)
        res = query_curser.captures(program)
        return res
        
    def match_query(self, program:tree_sitter.Node, query_str:str):
        query = Query(self.language, query_str)
        query_curser = QueryCursor(query)
        res = query_curser.matches(program)
        return [val for idx, val in res]
    
    def extract_function(self, program:tree_sitter.Node, target_modes:str=[], skip_external=False):
        query_str = '''
            (declaration_with_attrs
                (function_item)@function_item
            )@declaration_with_attrs
            '''
        declaration_matches = self.match_query(program, query_str)

        target_functions = []
        for match in declaration_matches:
            declaration = match['declaration_with_attrs'][0]
            function = match['function_item'][0]
            attributes = [child for child in declaration.children \
                          if child.type == 'attribute_item']
            if skip_external and any('verifier::external_body' in node_to_text(attr) for attr in attributes):
                continue
            function_modes = [node_to_text(child) for child in function.children \
                          if child.type == 'function_mode']
            if target_modes and len(set(function_modes) & set(target_modes))== 0:
                continue
            target_functions.append(declaration)
        return target_functions
vs_parser = verus_parser('/home/chentianyu/verus.so')

/tmp/ipykernel_3675595/1841071370.py:8: DeprecationWarning: int argument support is deprecated
  self.language = Language(ptr)


In [122]:
with open('../examples/mbpp_113.rs', 'r') as f:
    tmp_file = f.read()

In [150]:
root = vs_parser.parser.parse(bytes(tmp_file, encoding='utf-8'))

In [151]:
vs_parser.extract_function(root, target_modes=[], skip_external=False)

[<Node type=declaration_with_attrs, start_point=(2, 0), end_point=(8, 1)>,
 <Node type=declaration_with_attrs, start_point=(12, 0), end_point=(14, 1)>,
 <Node type=declaration_with_attrs, start_point=(16, 0), end_point=(21, 1)>,
 <Node type=declaration_with_attrs, start_point=(23, 0), end_point=(43, 1)>]

In [105]:
query_str = '''
(declaration_with_attrs
    (function_item)@function_item
)@declaration_with_attrs
'''

declaration_with_attrs_query = Query(vs_parser.language, query_str)
query_curser = QueryCursor(declaration_with_attrs_query)
query_curser.matches(root.root_node)

[(0,
  {'declaration_with_attrs': [<Node type=declaration_with_attrs, start_point=(2, 0), end_point=(8, 1)>],
   'function_item': [<Node type=function_item, start_point=(2, 0), end_point=(8, 1)>]}),
 (0,
  {'declaration_with_attrs': [<Node type=declaration_with_attrs, start_point=(12, 0), end_point=(14, 1)>],
   'function_item': [<Node type=function_item, start_point=(12, 0), end_point=(14, 1)>]}),
 (0,
  {'declaration_with_attrs': [<Node type=declaration_with_attrs, start_point=(16, 0), end_point=(21, 1)>],
   'function_item': [<Node type=function_item, start_point=(16, 0), end_point=(21, 1)>]}),
 (0,
  {'declaration_with_attrs': [<Node type=declaration_with_attrs, start_point=(23, 0), end_point=(43, 1)>],
   'function_item': [<Node type=function_item, start_point=(27, 4), end_point=(43, 1)>]})]

In [107]:
query_curser.matches(root.root_node)[1][1]['function_item'][0].children

[<Node type=function_mode, start_point=(12, 0), end_point=(12, 4)>,
 <Node type="fn", start_point=(12, 5), end_point=(12, 7)>,
 <Node type=identifier, start_point=(12, 8), end_point=(12, 21)>,
 <Node type=parameters, start_point=(12, 21), end_point=(12, 28)>,
 <Node type="->", start_point=(12, 29), end_point=(12, 31)>,
 <Node type=named_return_type, start_point=(12, 32), end_point=(12, 36)>,
 <Node type=block, start_point=(12, 37), end_point=(14, 1)>]

In [ ]:
query_curser.captures(root.root_node)['declaration_with_attrs'][0].text

In [60]:
query_curser = QueryCursor(declaration_with_attrs_query)

In [61]:
query_curser.captures(root.root_node)

{'declaration_with_attrs': [<Node type=declaration_with_attrs, start_point=(0, 0), end_point=(0, 21)>,
  <Node type=declaration_with_attrs, start_point=(16, 0), end_point=(21, 1)>,
  <Node type=declaration_with_attrs, start_point=(2, 0), end_point=(8, 1)>,
  <Node type=declaration_with_attrs, start_point=(23, 0), end_point=(39, 1)>,
  <Node type=declaration_with_attrs, start_point=(12, 0), end_point=(14, 1)>,
  <Node type=declaration_with_attrs, start_point=(27, 4), end_point=(27, 22)>]}

In [156]:
if [] == []:
    print('wow')

wow


In [67]:
root.root_node.children[2].children[3].children[0]

<Node type=function_item, start_point=(12, 0), end_point=(14, 1)>

In [5]:
tree_sitter

<module 'tree_sitter' from '/home/chentianyu/miniconda3/lib/python3.13/site-packages/tree_sitter/__init__.py'>